# 01 - Province Quarter Summary (Gold)

One row per canonical province, year, and quarter.
Production is summed. Climate fields are averaged (they repeat on every crop row).

In [0]:
from pyspark.sql import functions as F

from configs.project_config import SILVER_TABLES, GOLD_TABLES

source = SILVER_TABLES['crop_production_enriched']
target = GOLD_TABLES['province_quarter_summary']

df = spark.table(source)

summary = (
    df.groupBy('canonical_province', 'year', 'quarter')
    .agg(
        F.round(F.sum('production'), 3).alias('total_production'),
        F.round(F.sum(F.when(F.col('crop_group') == 'PALAY', F.col('production'))), 3).alias('palay_production'),
        F.round(F.sum(F.when(F.col('crop_group') == 'CORN', F.col('production'))), 3).alias('corn_production'),
        F.round(F.sum(F.when(F.col('crop_group') == 'OTHER', F.col('production'))), 3).alias('other_production'),
        F.round(F.avg('temperature_2m_mean'), 2).alias('avg_temperature'),
        F.round(F.avg('precipitation_sum'), 2).alias('total_rainfall'),
        F.round(F.avg('climate_stress'), 3).alias('climate_stress'),
        F.count(F.when(F.col('risk_label') == 'Stable', 1)).alias('stable_count'),
        F.count(F.when(F.col('risk_label') == 'Risk-prone', 1)).alias('risk_prone_count'),
        F.count(F.when(F.col('risk_label') == 'High', 1)).alias('high_count'),
        F.count(F.when(F.col('risk_label') == 'Declining', 1)).alias('declining_count'),
        F.countDistinct('crop').alias('crop_count')
    )
    .withColumnRenamed('canonical_province', 'province')
    .fillna(0, subset=['palay_production', 'corn_production', 'other_production'])
)

(
    summary.write
    .format('delta')
    .mode('overwrite')
    .option('overwriteSchema', 'true')
    .saveAsTable(target)
)

out = spark.table(target)
print(f'Wrote: {target}')
print(f'Rows : {out.count()}')
print(f'Cols : {out.columns}')